# EY — Fast LightGBM + Target Engineering
- **Alkalinity**: fixed params, no tuning, ~1 min
- **EC + DRP**: 10 Optuna trials × 3 folds, ~5 min each
- **Total**: ~12 min on laptop CPU

In [1]:
import pandas as pd
import numpy as np
import os, json, warnings, optuna
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
EPS = 1e-6

class Config:
    BASE_DIR = './data'
    SEED = 85
    N_FOLDS = 3          # 3 folds not 5
    N_TRIALS = 10        # 10 trials not 30
    TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\knich\miniconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\knich\miniconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\knich\miniconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\knich\miniconda3\Lib\site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\knich\miniconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\knich\miniconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\knich\miniconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\knich\miniconda3\Lib\site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
# ==========================================
# ALL FEATURE ENGINEERING IN ONE CELL
# ==========================================
def engineer_all(df, target=None):
    d = df.copy()
    
    # --- Shared (all targets) ---
    if all(c in d.columns for c in ['nir','green','swir16','swir22']):
        d['nir_green_ratio'] = d['nir'] / (d['green'] + EPS)
        d['swir16_nir_ratio'] = d['swir16'] / (d['nir'] + EPS)
        d['swir22_nir_ratio'] = d['swir22'] / (d['nir'] + EPS)
        d['swir16_green_ratio'] = d['swir16'] / (d['green'] + EPS)
        d['nir_minus_green'] = d['nir'] - d['green']
        d['swir_diff'] = d['swir16'] - d['swir22']
    for f in ['Popdens_00','SOC','dist_km']:
        if f in d.columns: d[f'log_{f}'] = np.log1p(d[f])
    
    # --- EC-specific ---
    if target == 'Electrical Conductance':
        if all(c in d.columns for c in ['pet','wet_season']):
            d['ec_pet_x_dry'] = d['pet'] * (1 - d['wet_season'])
            d['ec_pet_x_wet'] = d['pet'] * d['wet_season']
        if all(c in d.columns for c in ['pet','month_sin','month_cos']):
            d['ec_pet_x_msin'] = d['pet'] * d['month_sin']
            d['ec_pet_x_mcos'] = d['pet'] * d['month_cos']
        if all(c in d.columns for c in ['swir16','swir22','nir','green']):
            d['ec_salinity_idx'] = np.sqrt(np.abs(d['swir16'] * d['swir22']))
            d['ec_ndsi'] = (d['swir16'] - d['swir22']) / (d['swir16'] + d['swir22'] + EPS)
            d['ec_brightness'] = np.sqrt(d['green']**2 + d['nir']**2 + d['swir16']**2)
        ions = [c for c in ['dws_Ca','dws_Mg','dws_Na','dws_Cl','dws_SO4'] if c in d.columns]
        if len(ions) >= 2:
            d['ec_total_ions'] = d[ions].sum(axis=1)
        if all(c in d.columns for c in ['dws_Cl','dws_SO4']):
            d['ec_cl_so4_ratio'] = d['dws_Cl'] / (d['dws_SO4'] + EPS)
        if all(c in d.columns for c in ['dws_Ca','dws_Mg','dws_Na']):
            d['ec_camg_na_ratio'] = (d['dws_Ca']+d['dws_Mg']) / (d['dws_Na'] + EPS)
        if all(c in d.columns for c in ['Soil_pH','pet']):
            d['ec_soilpH_x_pet'] = d['Soil_pH'] * d['pet']
        if all(c in d.columns for c in ['Soil_wetness','pet']):
            d['ec_drysoil_pet'] = (1 - d['Soil_wetness']) * d['pet']
        if all(c in d.columns for c in ['sc','pet']):
            d['ec_sc_x_pet'] = d['sc'] * d['pet']
        if all(c in d.columns for c in ['dws_EC','dws_dist_km','dws_days_diff']):
            d['ec_dws_decay'] = d['dws_EC'] / (1 + d['dws_dist_km']*0.1 + np.abs(d['dws_days_diff'])*0.01)
    
    # --- DRP-specific ---
    if target == 'Dissolved Reactive Phosphorus':
        if all(c in d.columns for c in ['GLC_Managed','wet_season']):
            d['drp_agri_x_wet'] = d['GLC_Managed'] * d['wet_season']
            d['drp_agri_x_dry'] = d['GLC_Managed'] * (1 - d['wet_season'])
        if all(c in d.columns for c in ['GLC_Managed','pet']):
            d['drp_agri_x_pet'] = d['GLC_Managed'] * d['pet']
        if all(c in d.columns for c in ['GLC_Managed','NDMI']):
            d['drp_agri_x_ndmi'] = d['GLC_Managed'] * d['NDMI']
        if all(c in d.columns for c in ['GLC_Artificial','Popdens_00']):
            d['drp_human_pressure'] = np.log1p(d['Popdens_00']) * d['GLC_Artificial']
        if all(c in d.columns for c in ['GLC_Artificial','wet_season']):
            d['drp_urban_x_wet'] = d['GLC_Artificial'] * d['wet_season']
        if all(c in d.columns for c in ['Popdens_00','wet_season']):
            d['drp_pop_x_wet'] = np.log1p(d['Popdens_00']) * d['wet_season']
        if all(c in d.columns for c in ['NDMI','MNDWI']):
            d['drp_water_signal'] = d['MNDWI'] * d['NDMI']
        if all(c in d.columns for c in ['MNDWI','wet_season']):
            d['drp_mndwi_x_wet'] = d['MNDWI'] * d['wet_season']
        if all(c in d.columns for c in ['nir','green']):
            d['drp_chl_proxy'] = d['nir'] / (d['green'] + EPS)
            d['drp_green_dominance'] = d['green'] / (d['nir'] + d['green'] + EPS)
        if all(c in d.columns for c in ['nir','swir16']):
            d['drp_turbidity_proxy'] = (d['nir']-d['swir16']) / (d['nir']+d['swir16'] + EPS)
        if all(c in d.columns for c in ['GLC_Aquatic_Veg','wet_season']):
            d['drp_aquaveg_x_wet'] = d['GLC_Aquatic_Veg'] * d['wet_season']
        if all(c in d.columns for c in ['dws_P_modified','dws_dist_km','dws_days_diff']):
            d['drp_dws_decay'] = d['dws_P_modified'] / (1 + d['dws_dist_km']*0.1 + np.abs(d['dws_days_diff'])*0.01)
        if 'month' in d.columns:
            d['drp_peak_flush'] = d['month'].isin([12,1,2,3]).astype(int)
            d['drp_first_flush'] = d['month'].isin([10,11]).astype(int)
        if 'pet' in d.columns:
            d['drp_inv_pet'] = 1.0 / (d['pet'] + EPS)
    
    return d

In [3]:
# ==========================================
# LOAD + CLUSTER
# ==========================================
train_full = pd.read_csv(f'{Config.BASE_DIR}/train_ALL+reliability.csv')
test_full = pd.read_csv(f'{Config.BASE_DIR}/test_ALL+reliability.csv')

stations = train_full.groupby(['Latitude','Longitude']).size().reset_index()
km = KMeans(n_clusters=Config.N_FOLDS, random_state=Config.SEED, n_init=10)
stations['spatial_cluster'] = km.fit_predict(stations[['Latitude','Longitude']])
train_full = train_full.merge(stations[['Latitude','Longitude','spatial_cluster']], on=['Latitude','Longitude'], how='left')
groups = train_full['spatial_cluster']

ignore = Config.TARGETS + ['Latitude','Longitude','STAT_ID','Sample Date','spatial_cluster',
    'geometry','_merge_terra','_merge_landsat','Latitude_glorich','Longitude_glorich',
    'date','dws_1st','Impute_Method','P_modified_same','dws_P_reliability']

def make_X(df, target):
    d = engineer_all(df, target=target)
    feats = [c for c in d.columns if c not in ignore]
    X = d[feats].select_dtypes(include=[np.number])
    X = X.loc[:, X.nunique() > 1]
    return X

print(f"Loaded: train={len(train_full)}, test={len(test_full)}")

Loaded: train=9319, test=200


In [4]:
# ==========================================
# ALKALINITY — NO TUNING, fixed good params
# ==========================================
import time
t0 = time.time()

target = 'Total Alkalinity'
print(f"--- {target} (fixed params, no Optuna) ---")

X_tr = make_X(train_full, target)
X_te = make_X(test_full, target)
X_te = X_te[[c for c in X_tr.columns if c in X_te.columns]]
X_tr = X_tr[X_te.columns]  # align

y_alk = train_full[target]

alk_params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 800, 'learning_rate': 0.03, 'num_leaves': 63,
    'max_depth': 6, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'subsample_freq': 1, 'verbosity': -1, 'seed': Config.SEED, 'n_jobs': -1,
}

# Quick CV score check (single pass, no tuning)
gkf = GroupKFold(n_splits=Config.N_FOLDS)
alk_scores = []
for tr_i, va_i in gkf.split(X_tr, y_alk, groups=groups):
    m = lgb.LGBMRegressor(**alk_params)
    m.fit(X_tr.iloc[tr_i], y_alk.iloc[tr_i],
          eval_set=[(X_tr.iloc[va_i], y_alk.iloc[va_i])],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    alk_scores.append(r2_score(y_alk.iloc[va_i], m.predict(X_tr.iloc[va_i])))

print(f"  CV R²: {np.mean(alk_scores):.4f} ({time.time()-t0:.0f}s)")

# Final model
alk_model = lgb.LGBMRegressor(**alk_params)
alk_model.fit(X_tr, y_alk)
alk_preds = np.clip(alk_model.predict(X_te), 0, None)

print(f"  Done. {X_tr.shape[1]} features, {time.time()-t0:.0f}s total")

--- Total Alkalinity (fixed params, no Optuna) ---
  CV R²: 0.9771 (18s)
  Done. 59 features, 28s total


In [5]:
# ==========================================
# EC + DRP — Light Optuna (10 trials × 3 folds)
# ==========================================
def fast_objective(trial, X, y, groups, target_name):
    param = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'verbosity': -1, 'seed': Config.SEED, 'n_jobs': -1,
        'n_estimators': 600,  # lighter
        'learning_rate': trial.suggest_float('lr', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('nl', 20, 100),
        'max_depth': trial.suggest_int('md', 4, 7),
        'reg_alpha': trial.suggest_float('ra', 0.01, 5.0, log=True),
        'reg_lambda': trial.suggest_float('rl', 0.01, 5.0, log=True),
        'min_child_samples': trial.suggest_int('mcs', 10, 80),
        'subsample': trial.suggest_float('ss', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('cs', 0.6, 1.0),
        'subsample_freq': 1,
    }
    if 'Phosphorus' in target_name:
        if trial.suggest_categorical('huber', [True, False]):
            param['objective'] = 'huber'
            param['alpha'] = trial.suggest_float('ha', 1.0, 15.0)
    
    gkf = GroupKFold(n_splits=Config.N_FOLDS)
    scores = []
    for tr_i, va_i in gkf.split(X, y, groups=groups):
        m = lgb.LGBMRegressor(**param)
        m.fit(X.iloc[tr_i], y.iloc[tr_i],
              eval_set=[(X.iloc[va_i], y.iloc[va_i])],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
        p = m.predict(X.iloc[va_i])
        if 'Phosphorus' in target_name:
            scores.append(r2_score(np.expm1(y.iloc[va_i]), np.expm1(p)))
        else:
            scores.append(r2_score(y.iloc[va_i], p))
    return np.mean(scores)

ec_drp_models = {}
ec_drp_preds = {}
cv_scores = {'Total Alkalinity': np.mean(alk_scores)}

for target in ['Electrical Conductance', 'Dissolved Reactive Phosphorus']:
    t0 = time.time()
    print(f"\n--- {target} (10 trials × 3 folds) ---")
    
    X_tr = make_X(train_full, target)
    X_te = make_X(test_full, target)
    X_te = X_te[[c for c in X_tr.columns if c in X_te.columns]]
    X_tr = X_tr[X_te.columns]
    
    y = train_full[target].copy()
    y_train = np.log1p(y) if 'Phosphorus' in target else y
    y_train.name = target
    
    study = optuna.create_study(direction='maximize')
    study.optimize(
        lambda trial: fast_objective(trial, X_tr, y_train, groups, target),
        n_trials=Config.N_TRIALS, show_progress_bar=True
    )
    cv_scores[target] = study.best_value
    print(f"  Best CV R²: {study.best_value:.4f}")
    
    # Final model with best params
    bp = {k: v for k, v in study.best_params.items() if k not in ['huber','ha']}
    # Remap short names back
    name_map = {'lr':'learning_rate','nl':'num_leaves','md':'max_depth',
                'ra':'reg_alpha','rl':'reg_lambda','mcs':'min_child_samples',
                'ss':'subsample','cs':'colsample_bytree'}
    bp = {name_map.get(k,k): v for k,v in bp.items()}
    bp.update({'n_estimators': 1000, 'objective': 'regression', 'metric': 'rmse',
               'verbosity': -1, 'seed': Config.SEED, 'n_jobs': -1, 'subsample_freq': 1})
    if study.best_params.get('huber'):
        bp['objective'] = 'huber'
        bp['alpha'] = study.best_params.get('ha', 9.0)
    
    fm = lgb.LGBMRegressor(**bp)
    fm.fit(X_tr, y_train)
    ec_drp_models[target] = fm
    
    p = fm.predict(X_te)
    if 'Phosphorus' in target: p = np.expm1(p)
    ec_drp_preds[target] = np.clip(p, 0, None)
    
    # Quick importance peek
    prefix = 'ec_' if 'Conductance' in target else 'drp_'
    fi = pd.Series(fm.booster_.feature_importance(importance_type='gain'), index=X_tr.columns)
    fi = fi.sort_values(ascending=False)
    target_fi = fi[fi.index.str.startswith(prefix)]
    print(f"  {X_tr.shape[1]} features, {len(target_fi)} target-specific")
    print(f"  Target-specific gain share: {target_fi.sum()/fi.sum()*100:.1f}%")
    print(f"  Top 5: {fi.head(5).index.tolist()}")
    print(f"  {time.time()-t0:.0f}s")


--- Electrical Conductance (10 trials × 3 folds) ---


  0%|          | 0/10 [00:00<?, ?it/s]

  Best CV R²: 0.9118
  72 features, 13 target-specific
  Target-specific gain share: 52.5%
  Top 5: ['ec_dws_decay', 'dws_EC', 'Soil_pH', 'SpecCond25C', 'pet']
  140s

--- Dissolved Reactive Phosphorus (10 trials × 3 folds) ---


  0%|          | 0/10 [00:00<?, ?it/s]

  Best CV R²: 0.6740
  75 features, 16 target-specific
  Target-specific gain share: 71.7%
  Top 5: ['drp_dws_decay', 'dws_P_modified', 'dws_Na', 'DIP', 'year']
  155s


In [9]:
# ==========================================
# SUBMISSION
# ==========================================
submission = pd.DataFrame({
    'Latitude': test_full['Latitude'],
    'Longitude': test_full['Longitude'],
    'Sample Date': test_full['Sample Date'],
    'Total Alkalinity': alk_preds,
    'Electrical Conductance': ec_drp_preds['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': ec_drp_preds['Dissolved Reactive Phosphorus']
})

dws_tal = test_full['dws_TAL'].values
submission['Total Alkalinity'] = np.where(
    np.isfinite(dws_tal), dws_tal, submission['Total Alkalinity']
)

submission.to_csv(f'{Config.BASE_DIR}/submission_lgbm_fast.csv', index=False)

# Ensemble with CatBoost if available
cat_path = f'{Config.BASE_DIR}/submission_reliabilityOfDWS_temp1_all_features.csv'
if os.path.exists(cat_path):
    sub_cat = pd.read_csv(cat_path)
    ens = sub_cat[['Latitude','Longitude','Sample Date']].copy()
    for t in Config.TARGETS:
        ens[t] = np.clip(0.5*sub_cat[t] + 0.5*submission[t], 0, None)
    ens.to_csv(f'{Config.BASE_DIR}/submission_ensemble_fast.csv', index=False)
    print("Ensemble saved!")

print(f"\n--- SCORES ---")
for t,s in cv_scores.items(): print(f"  {t}: {s:.4f}")
print(f"  MEAN: {np.mean(list(cv_scores.values())):.4f}")

Ensemble saved!

--- SCORES ---
  Total Alkalinity: 0.9771
  Electrical Conductance: 0.9118
  Dissolved Reactive Phosphorus: 0.6740
  MEAN: 0.8543


In [8]:
# ens.to_csv(f'{Config.BASE_DIR}/submission_ensemble_fast.csv', index=False)
# print("Ensemble saved!")

ens.describe()

NameError: name 'ens' is not defined

In [10]:
"""
45-MIN EMERGENCY SCRIPT
Run this once → get 5 submission CSVs → upload all of them
"""
import pandas as pd
import numpy as np

# ========== CHANGE THESE PATHS ==========
BASE = './data'
CAT_PATH = f'{BASE}/submission_reliabilityOfDWS_temp1_all_features.csv'
LGB_PATH = f'{BASE}/submission_lgbm_fast.csv'
TEST_PATH = f'{BASE}/test_ALL+reliability.csv'
# =========================================

cat = pd.read_csv(CAT_PATH)
lgb = pd.read_csv(LGB_PATH)
test = pd.read_csv(TEST_PATH)

meta = cat[['Latitude', 'Longitude', 'Sample Date']].copy()
targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# ------------------------------------------
# 1. SIMPLE 50/50 ENSEMBLE (submit first as safety)
# ------------------------------------------
ens50 = meta.copy()
for t in targets:
    ens50[t] = np.clip(0.5 * cat[t] + 0.5 * lgb[t], 0, None)
ens50.to_csv(f'{BASE}/SUB_ensemble_50_50.csv', index=False)
print("✅ 1/5: 50/50 ensemble saved")

# ------------------------------------------
# 2. DWS OVERRIDE FOR ALL 3 TARGETS (big potential lift)
# ------------------------------------------
# Unit conversions: DWS EC is mS/m → competition is µS/cm (×10)
#                   DWS PO4 is mg/L → competition is µg/L (×1000)

def apply_dws_override(sub, test_df, max_dist_km=5, max_days=30):
    """Override predictions with DWS measurements when station is close + recent."""
    out = sub.copy()
    
    dist = test_df['dws_dist_km'].values if 'dws_dist_km' in test_df.columns else np.full(len(test_df), np.inf)
    days = np.abs(test_df['dws_days_diff'].values) if 'dws_days_diff' in test_df.columns else np.full(len(test_df), np.inf)
    
    close_mask = (dist <= max_dist_km) & (days <= max_days)
    
    # TAL — no conversion needed
    if 'dws_TAL' in test_df.columns:
        dws_tal = test_df['dws_TAL'].values
        mask = close_mask & np.isfinite(dws_tal)
        out.loc[mask, 'Total Alkalinity'] = dws_tal[mask]
        print(f"   TAL: overrode {mask.sum()} rows")
    
    # EC — DWS is mS/m, competition is µS/cm → multiply by 10
    if 'dws_EC' in test_df.columns:
        dws_ec = test_df['dws_EC'].values * 10
        mask = close_mask & np.isfinite(dws_ec)
        out.loc[mask, 'Electrical Conductance'] = dws_ec[mask]
        print(f"   EC:  overrode {mask.sum()} rows")
    
    # DRP — DWS PO4 is mg/L, competition is µg/L → multiply by 1000
    if 'dws_P_modified' in test_df.columns:
        dws_drp = test_df['dws_P_modified'].values * 1000
        mask = close_mask & np.isfinite(dws_drp)
        out.loc[mask, 'Dissolved Reactive Phosphorus'] = dws_drp[mask]
        print(f"   DRP: overrode {mask.sum()} rows")
    
    return out

# Apply DWS override on the 50/50 ensemble
ens_dws = apply_dws_override(ens50, test)
ens_dws.to_csv(f'{BASE}/SUB_ensemble_50_50_dws_all.csv', index=False)
print("✅ 2/5: 50/50 + DWS override (all targets) saved")

# ------------------------------------------
# 3. DOMAIN CLIPPING (realistic ranges from guidance)
# ------------------------------------------
def clip_domain(sub):
    out = sub.copy()
    out['Total Alkalinity'] = out['Total Alkalinity'].clip(0, 400)
    out['Electrical Conductance'] = out['Electrical Conductance'].clip(0, 1600)
    out['Dissolved Reactive Phosphorus'] = out['Dissolved Reactive Phosphorus'].clip(0, 200)
    return out

ens_dws_clip = clip_domain(ens_dws)
ens_dws_clip.to_csv(f'{BASE}/SUB_ensemble_dws_clipped.csv', index=False)
print("✅ 3/5: 50/50 + DWS + clipped saved")

# ------------------------------------------
# 4. TRY DIFFERENT BLEND WEIGHTS
# ------------------------------------------
# 60/40 favoring CatBoost (if CatBoost had better CV)
ens60 = meta.copy()
for t in targets:
    ens60[t] = np.clip(0.6 * cat[t] + 0.4 * lgb[t], 0, None)
ens60 = apply_dws_override(ens60, test)
ens60 = clip_domain(ens60)
ens60.to_csv(f'{BASE}/SUB_ensemble_60cat_40lgb.csv', index=False)
print("✅ 4/5: 60/40 CatBoost-heavy + DWS + clip saved")

# 40/60 favoring LightGBM (if LightGBM had better CV)
ens40 = meta.copy()
for t in targets:
    ens40[t] = np.clip(0.4 * cat[t] + 0.6 * lgb[t], 0, None)
ens40 = apply_dws_override(ens40, test)
ens40 = clip_domain(ens40)
ens40.to_csv(f'{BASE}/SUB_ensemble_40cat_60lgb.csv', index=False)
print("✅ 5/5: 40/60 LightGBM-heavy + DWS + clip saved")

# ------------------------------------------
# SUMMARY
# ------------------------------------------
print("\n" + "="*50)
print("UPLOAD ORDER (best first):")
print("="*50)
print("1. SUB_ensemble_dws_clipped.csv       ← most likely best")
print("2. SUB_ensemble_60cat_40lgb.csv        ← if CatBoost was better")
print("3. SUB_ensemble_40cat_60lgb.csv        ← if LightGBM was better")
print("4. SUB_ensemble_50_50_dws_all.csv      ← without clipping")
print("5. SUB_ensemble_50_50.csv              ← vanilla fallback")

# Quick sanity check
print("\n--- Sanity Check ---")
for name, sub in [('50/50+DWS+clip', ens_dws_clip), ('60cat', ens60), ('40cat', ens40)]:
    print(f"\n{name}:")
    for t in targets:
        print(f"  {t}: mean={sub[t].mean():.1f}, min={sub[t].min():.1f}, max={sub[t].max():.1f}")

✅ 1/5: 50/50 ensemble saved
   TAL: overrode 200 rows
   EC:  overrode 200 rows
   DRP: overrode 200 rows
✅ 2/5: 50/50 + DWS override (all targets) saved
✅ 3/5: 50/50 + DWS + clipped saved
   TAL: overrode 200 rows
   EC:  overrode 200 rows
   DRP: overrode 200 rows
✅ 4/5: 60/40 CatBoost-heavy + DWS + clip saved
   TAL: overrode 200 rows
   EC:  overrode 200 rows
   DRP: overrode 200 rows
✅ 5/5: 40/60 LightGBM-heavy + DWS + clip saved

UPLOAD ORDER (best first):
1. SUB_ensemble_dws_clipped.csv       ← most likely best
2. SUB_ensemble_60cat_40lgb.csv        ← if CatBoost was better
3. SUB_ensemble_40cat_60lgb.csv        ← if LightGBM was better
4. SUB_ensemble_50_50_dws_all.csv      ← without clipping
5. SUB_ensemble_50_50.csv              ← vanilla fallback

--- Sanity Check ---

50/50+DWS+clip:
  Total Alkalinity: mean=124.6, min=17.9, max=400.0
  Electrical Conductance: mean=536.0, min=107.9, max=1600.0
  Dissolved Reactive Phosphorus: mean=27.1, min=10.0, max=200.0

60cat:
  Total A

In [11]:
"""
25-MIN DEADLINE SCRIPT
1. Load existing TAL + EC models → predict
2. Train FRESH DRP model with domain features (fixed params, no Optuna, ~3 min)
3. Also train a quick "no-DWS" EC model as backup (~2 min)
4. Blend everything → 4 submissions
"""
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
import time, warnings
warnings.filterwarnings('ignore')

# ========== CHANGE THESE ==========
BASE = './data'
TEST_PATH = f'{BASE}/test_ALL+reliability.csv'
TRAIN_PATH = f'{BASE}/train_ALL+reliability.csv'
TAL_MODEL = f'{BASE}/lgbm_Total_Alkalinity.txt'
EC_MODEL = f'{BASE}/lgbm_pruned_Electrical_Conductance.txt'
DRP_MODEL = f'{BASE}/lgbm_Dissolved_Reactive_Phosphorus.txt'
# ===================================

EPS = 1e-6
SEED = 85
t_start = time.time()

print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# Spatial clusters (3 folds for speed)
stations = train.groupby(['Latitude','Longitude']).size().reset_index()
km = KMeans(n_clusters=3, random_state=SEED, n_init=10)
stations['spatial_cluster'] = km.fit_predict(stations[['Latitude','Longitude']])
train = train.merge(stations[['Latitude','Longitude','spatial_cluster']], on=['Latitude','Longitude'], how='left')

# ==========================================
# SHARED FEATURE ENGINEERING
# ==========================================
def engineer(df):
    d = df.copy()
    if all(c in d.columns for c in ['nir','green','swir16','swir22']):
        d['nir_green_ratio'] = d['nir'] / (d['green'] + EPS)
    for f in ['Popdens_00','dist_km']:
        if f in d.columns:
            d[f'log_{f}'] = np.log1p(d[f])
    return d

def engineer_drp(df):
    """DRP domain features — ag runoff, sewage, flushing."""
    d = df.copy()
    if all(c in d.columns for c in ['GLC_Managed','wet_season']):
        d['drp_agri_x_wet'] = d['GLC_Managed'] * d['wet_season']
    if all(c in d.columns for c in ['GLC_Managed','pet']):
        d['drp_agri_x_pet'] = d['GLC_Managed'] * d['pet']
    if all(c in d.columns for c in ['GLC_Managed','NDMI']):
        d['drp_agri_x_ndmi'] = d['GLC_Managed'] * d['NDMI']
    if all(c in d.columns for c in ['GLC_Artificial','Popdens_00']):
        d['drp_human_pressure'] = np.log1p(d['Popdens_00']) * d['GLC_Artificial']
    if all(c in d.columns for c in ['GLC_Artificial','wet_season']):
        d['drp_urban_x_wet'] = d['GLC_Artificial'] * d['wet_season']
    if all(c in d.columns for c in ['Popdens_00','wet_season']):
        d['drp_pop_x_wet'] = np.log1p(d['Popdens_00']) * d['wet_season']
    if all(c in d.columns for c in ['NDMI','MNDWI']):
        d['drp_water_signal'] = d['MNDWI'] * d['NDMI']
    if all(c in d.columns for c in ['MNDWI','wet_season']):
        d['drp_mndwi_x_wet'] = d['MNDWI'] * d['wet_season']
    if all(c in d.columns for c in ['nir','green']):
        d['drp_chl_proxy'] = d['nir'] / (d['green'] + EPS)
        d['drp_green_dom'] = d['green'] / (d['nir'] + d['green'] + EPS)
    if all(c in d.columns for c in ['nir','swir16']):
        d['drp_turbidity'] = (d['nir']-d['swir16']) / (d['nir']+d['swir16'] + EPS)
    if 'month' in d.columns:
        d['drp_peak_flush'] = d['month'].isin([12,1,2,3]).astype(int)
        d['drp_first_flush'] = d['month'].isin([10,11]).astype(int)
    if 'pet' in d.columns:
        d['drp_inv_pet'] = 1.0 / (d['pet'] + EPS)
    if all(c in d.columns for c in ['GLC_Aquatic_Veg','wet_season']):
        d['drp_aquaveg_x_wet'] = d['GLC_Aquatic_Veg'] * d['wet_season']
    return d

def engineer_ec(df):
    """EC domain features — salinity, evaporation, ions."""
    d = df.copy()
    if all(c in d.columns for c in ['pet','wet_season']):
        d['ec_pet_x_dry'] = d['pet'] * (1 - d['wet_season'])
    if all(c in d.columns for c in ['swir16','swir22','nir','green']):
        d['ec_salinity_idx'] = np.sqrt(np.abs(d['swir16'] * d['swir22']))
        d['ec_brightness'] = np.sqrt(d['green']**2 + d['nir']**2 + d['swir16']**2)
    if all(c in d.columns for c in ['Soil_pH','pet']):
        d['ec_soilpH_x_pet'] = d['Soil_pH'] * d['pet']
    if all(c in d.columns for c in ['pet','month_sin','month_cos']):
        d['ec_pet_x_msin'] = d['pet'] * d['month_sin']
    return d

train_eng = engineer_drp(engineer_ec(engineer(train)))
test_eng = engineer_drp(engineer_ec(engineer(test)))

# ==========================================
# 1. TAL — Load existing model, predict
# ==========================================
print(f"\n[{time.time()-t_start:.0f}s] TAL: loading existing model...")
tal_model = lgb.Booster(model_file=TAL_MODEL)
tal_feats = tal_model.feature_name()
tal_preds = tal_model.predict(test_eng[tal_feats])
tal_preds = np.clip(tal_preds, 0, 400)
print(f"  TAL done. mean={np.mean(tal_preds):.1f}")

# ==========================================
# 2. EC — Load existing model + train fresh no-DWS backup
# ==========================================
print(f"\n[{time.time()-t_start:.0f}s] EC: loading existing pruned model...")
ec_model_old = lgb.Booster(model_file=EC_MODEL)
ec_feats_old = ec_model_old.feature_name()
ec_preds_old = ec_model_old.predict(test_eng[ec_feats_old])
ec_preds_old = np.clip(ec_preds_old, 0, 1600)
print(f"  EC (pruned) done. mean={np.mean(ec_preds_old):.1f}")

# Fresh EC model WITHOUT DWS features (better generalization)
print(f"[{time.time()-t_start:.0f}s] EC: training fresh no-DWS model...")
dws_cols = [c for c in train_eng.columns if c.startswith('dws_') or c in
    ['Alkalinity','Cl','DIP','SO4','SpecCond25C','pH',
     'Alkalinity_reliability','Cl_reliability','DIP_reliability',
     'SO4_reliability','SpecCond25C_reliability','pH_reliability',
     'date_diff_days','dws_1st_dist','dist_m','dist_km','log_dist_km',
     'Latitude_glorich','Longitude_glorich','date','P_modified_same','dws_P_reliability']]
ignore = ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus',
          'Latitude','Longitude','STAT_ID','Sample Date','spatial_cluster',
          'geometry','_merge_terra','_merge_landsat','Impute_Method','dws_1st'] + dws_cols
ec_feats_new = [c for c in train_eng.columns if c not in ignore and train_eng[c].dtype in ['float64','int64','float32','int32']]
ec_feats_new = [c for c in ec_feats_new if train_eng[c].nunique() > 1]
# Make sure test has them
ec_feats_new = [c for c in ec_feats_new if c in test_eng.columns]

ec_params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 600, 'learning_rate': 0.03, 'num_leaves': 63,
    'max_depth': 6, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.7,
    'subsample_freq': 1, 'verbosity': -1, 'seed': SEED, 'n_jobs': -1,
}
ec_new = lgb.LGBMRegressor(**ec_params)
ec_new.fit(train_eng[ec_feats_new], train_eng['Electrical Conductance'])
ec_preds_new = np.clip(ec_new.predict(test_eng[ec_feats_new]), 0, 1600)
print(f"  EC (no-DWS) done. {len(ec_feats_new)} feats, mean={np.mean(ec_preds_new):.1f}")

# ==========================================
# 3. DRP — Train FRESH with domain features, NO DWS
# ==========================================
print(f"\n[{time.time()-t_start:.0f}s] DRP: training fresh model with domain features...")

# DRP features: everything except DWS/GLORICH + add domain features
drp_feats = [c for c in train_eng.columns if c not in ignore and c.startswith('drp_') or
             (c not in ignore and train_eng[c].dtype in ['float64','int64','float32','int32']
              and c not in dws_cols)]
drp_feats = [c for c in drp_feats if train_eng[c].nunique() > 1 and c in test_eng.columns]
# Remove any targets or metadata that slipped through
drp_feats = [c for c in drp_feats if c not in ignore]

y_drp = np.log1p(train['Dissolved Reactive Phosphorus'])

drp_params = {
    'objective': 'huber', 'alpha': 9.0, 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 600, 'learning_rate': 0.02, 'num_leaves': 64,
    'max_depth': 5, 'reg_alpha': 0.3, 'reg_lambda': 1.0,
    'min_child_samples': 55, 'subsample': 0.65, 'colsample_bytree': 0.8,
    'subsample_freq': 1, 'verbosity': -1, 'seed': SEED, 'n_jobs': -1,
}

# Quick CV to see the score
gkf = GroupKFold(n_splits=3)
drp_scores = []
for tr_i, va_i in gkf.split(train_eng[drp_feats], y_drp, groups=train['spatial_cluster']):
    m = lgb.LGBMRegressor(**drp_params)
    m.fit(train_eng[drp_feats].iloc[tr_i], y_drp.iloc[tr_i],
          eval_set=[(train_eng[drp_feats].iloc[va_i], y_drp.iloc[va_i])],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    p = np.expm1(m.predict(train_eng[drp_feats].iloc[va_i]))
    a = np.expm1(y_drp.iloc[va_i])
    drp_scores.append(r2_score(a, p))
print(f"  DRP (no-DWS + domain) CV R²: {np.mean(drp_scores):.4f}")

# Final model
drp_new = lgb.LGBMRegressor(**drp_params)
drp_new.fit(train_eng[drp_feats], y_drp)
drp_preds_new = np.clip(np.expm1(drp_new.predict(test_eng[drp_feats])), 0, 200)
print(f"  DRP done. {len(drp_feats)} feats, mean={np.mean(drp_preds_new):.1f}")

# Also get old DRP predictions
drp_model_old = lgb.Booster(model_file=DRP_MODEL)
drp_feats_old = drp_model_old.feature_name()
drp_preds_old = np.clip(np.expm1(drp_model_old.predict(test_eng[drp_feats_old])), 0, 200)

# Top DRP features
fi = pd.Series(drp_new.booster_.feature_importance(importance_type='gain'), index=drp_feats)
print(f"  DRP top 10: {fi.sort_values(ascending=False).head(10).index.tolist()}")

# ==========================================
# 4. DWS OVERRIDE
# ==========================================
def dws_override(sub, test_df, max_dist=5, max_days=30):
    out = sub.copy()
    dist = test_df.get('dws_dist_km', pd.Series(np.inf, index=test_df.index)).values
    days = np.abs(test_df.get('dws_days_diff', pd.Series(np.inf, index=test_df.index)).values)
    close = (dist <= max_dist) & (days <= max_days)

    if 'dws_TAL' in test_df.columns:
        v = test_df['dws_TAL'].values
        mask = close & np.isfinite(v)
        out.loc[mask, 'Total Alkalinity'] = v[mask]

    if 'dws_EC' in test_df.columns:
        v = test_df['dws_EC'].values * 10  # mS/m → µS/cm
        mask = close & np.isfinite(v)
        out.loc[mask, 'Electrical Conductance'] = v[mask]

    # DRP: check if dws_P_modified is already in µg/L or mg/L
    if 'dws_P_modified' in test_df.columns:
        v = test_df['dws_P_modified'].values
        # If max < 1, it's in mg/L → convert
        if np.nanmax(v) < 10:
            v = v * 1000
        mask = close & np.isfinite(v)
        out.loc[mask, 'Dissolved Reactive Phosphorus'] = v[mask]

    return out

meta = pd.DataFrame({
    'Latitude': test['Latitude'],
    'Longitude': test['Longitude'],
    'Sample Date': test['Sample Date'],
})

# ==========================================
# 5. BUILD SUBMISSIONS
# ==========================================
print(f"\n[{time.time()-t_start:.0f}s] Building submissions...")

# SUB 1: Best guess — old TAL, new EC, new DRP (no-DWS models generalize better)
sub1 = meta.copy()
sub1['Total Alkalinity'] = tal_preds
sub1['Electrical Conductance'] = ec_preds_new
sub1['Dissolved Reactive Phosphorus'] = drp_preds_new
sub1 = dws_override(sub1, test)
sub1.to_csv(f'{BASE}/SUB1_new_ec_drp.csv', index=False)
print("✅ SUB1: old TAL + new EC + new DRP + DWS override")

# SUB 2: Blend old and new for EC and DRP
sub2 = meta.copy()
sub2['Total Alkalinity'] = tal_preds
sub2['Electrical Conductance'] = 0.4 * ec_preds_old + 0.6 * ec_preds_new
sub2['Dissolved Reactive Phosphorus'] = 0.4 * drp_preds_old + 0.6 * drp_preds_new
sub2 = dws_override(sub2, test)
sub2.to_csv(f'{BASE}/SUB2_blend_old_new.csv', index=False)
print("✅ SUB2: TAL + blended EC (40old/60new) + blended DRP (40old/60new)")

# SUB 3: All old models (safety net)
sub3 = meta.copy()
sub3['Total Alkalinity'] = tal_preds
sub3['Electrical Conductance'] = ec_preds_old
sub3['Dissolved Reactive Phosphorus'] = drp_preds_old
sub3 = dws_override(sub3, test)
sub3.to_csv(f'{BASE}/SUB3_all_old.csv', index=False)
print("✅ SUB3: all old models + DWS override")

# SUB 4: All new (maximum generalization)
sub4 = meta.copy()
sub4['Total Alkalinity'] = tal_preds
sub4['Electrical Conductance'] = ec_preds_new
sub4['Dissolved Reactive Phosphorus'] = drp_preds_new
# NO DWS override — pure model predictions
sub4.to_csv(f'{BASE}/SUB4_all_new_no_dws.csv', index=False)
print("✅ SUB4: new models, NO DWS override (pure generalization)")

# ==========================================
# SUMMARY
# ==========================================
elapsed = time.time() - t_start
print(f"\n{'='*50}")
print(f"  DONE in {elapsed:.0f}s")
print(f"{'='*50}")
print(f"  DRP no-DWS CV: {np.mean(drp_scores):.4f}")
print(f"\n  UPLOAD ORDER:")
print(f"  1. SUB1_new_ec_drp.csv         ← best bet")
print(f"  2. SUB2_blend_old_new.csv       ← hedged blend")
print(f"  3. SUB4_all_new_no_dws.csv      ← if DWS hurts on leaderboard")
print(f"  4. SUB3_all_old.csv             ← safety fallback")

for name, s in [('SUB1',sub1),('SUB2',sub2),('SUB3',sub3),('SUB4',sub4)]:
    print(f"\n  {name}:")
    for t in ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']:
        print(f"    {t}: mean={s[t].mean():.1f} min={s[t].min():.1f} max={s[t].max():.1f}")

Loading data...

[1s] TAL: loading existing model...
  TAL done. mean=119.1

[1s] EC: loading existing pruned model...
  EC (pruned) done. mean=499.7
[1s] EC: training fresh no-DWS model...
  EC (no-DWS) done. 53 feats, mean=453.5

[19s] DRP: training fresh model with domain features...
  DRP (no-DWS + domain) CV R²: -0.0630
  DRP done. 53 feats, mean=23.3
  DRP top 10: ['GLC_Artificial', 'drp_human_pressure', 'GLC_Managed', 'Popdens_00', 'drp_agri_x_pet', 'ss', 'pet', 'year', 'ec_soilpH_x_pet', 'drp_inv_pet']

[32s] Building submissions...
✅ SUB1: old TAL + new EC + new DRP + DWS override
✅ SUB2: TAL + blended EC (40old/60new) + blended DRP (40old/60new)
✅ SUB3: all old models + DWS override
✅ SUB4: new models, NO DWS override (pure generalization)

  DONE in 32s
  DRP no-DWS CV: -0.0630

  UPLOAD ORDER:
  1. SUB1_new_ec_drp.csv         ← best bet
  2. SUB2_blend_old_new.csv       ← hedged blend
  3. SUB4_all_new_no_dws.csv      ← if DWS hurts on leaderboard
  4. SUB3_all_old.csv     